# Nepali Corpus Collector — KAGGLE (final)
Everything: Wikimedia dumps + 34 news/government sources (parallel) + literature BFS + Archive.org deep sweep + live Wikimedia API fallback. No GPU needed — leave the accelerator off to save your GPU quota for OCR notebooks.

**Durability note:** Kaggle has no Drive to sync to, but `/kaggle/working` is live and downloadable from the Output panel (right sidebar) at any point while the session is running — you don't need to wait for the notebook to finish or click Save Version. Check in periodically and download the .txt file as a manual backup.

In [ ]:
!pip install -q requests beautifulsoup4 lxml tqdm

## Cell 1: Setup

In [ ]:
import os, re, json, time, bz2, threading, requests, traceback
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, quote
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import deque
from tqdm import tqdm

# ── Kaggle setup: /kaggle/working persists live while the session runs, and is
# downloadable from the Output panel at any time WITHOUT needing to commit/Save Version.
# Check that panel periodically and download the .txt as a manual safety net -- if the
# whole kernel gets killed (not just disconnected), anything not yet flushed is at risk,
# same as any other environment.
OUT_DIR = '/kaggle/working/nepali_web_crawl_output'
os.makedirs(OUT_DIR, exist_ok=True)
TXT_FILE   = os.path.join(OUT_DIR, 'raw_web_crawled_nepali.txt')
CKPT_FILE  = os.path.join(OUT_DIR, 'crawled_urls.json')
FAIL_FILE  = os.path.join(OUT_DIR, 'failed_urls.json')
STATS_FILE = os.path.join(OUT_DIR, 'web_crawl_stats.json')

DISCOVERY_WORKERS = 15
FETCH_WORKERS      = 15


HEADERS = {'User-Agent': 'NepaliCorpusBot/1.0 (academic)', 'Accept-Language': 'ne,en;q=0.8'}
state_lock = threading.Lock()
file_lock  = threading.Lock()
domain_ts  = {}
_save_ctr  = 0

def load_ckpt():
    try:
        done = set(json.load(open(CKPT_FILE, encoding='utf-8'))) if os.path.exists(CKPT_FILE) else set()
    except Exception:
        done = set()
    try:
        fail = set(json.load(open(FAIL_FILE, encoding='utf-8'))) if os.path.exists(FAIL_FILE) else set()
    except Exception:
        fail = set()
    return done, fail

def flush_ckpt(done, fail, stats):
    with state_lock:
        try: json.dump(sorted(done),  open(CKPT_FILE,  'w', encoding='utf-8'), ensure_ascii=False)
        except Exception: pass
        try: json.dump(sorted(fail),  open(FAIL_FILE,  'w', encoding='utf-8'), ensure_ascii=False)
        except Exception: pass
        try: json.dump(stats,         open(STATS_FILE, 'w', encoding='utf-8'), ensure_ascii=False)
        except Exception: pass

def load_stats():
    try:
        return json.load(open(STATS_FILE, encoding='utf-8')) if os.path.exists(STATS_FILE) else {}
    except Exception:
        return {}

def est_tokens(stats):
    return int(sum(s.get('w', 0) for s in stats.values()) * 1.3)

def throttle(url, delay=0.15):
    d = urlparse(url).netloc
    with state_lock:
        wait = delay - (time.time() - domain_ts.get(d, 0))
        if wait > 0: time.sleep(wait)
        domain_ts[d] = time.time()

def ne_ratio(txt):
    if not txt: return 0.0
    dev = sum(1 for c in txt if '\u0900' <= c <= '\u097F')
    ns = sum(1 for c in txt if not c.isspace())
    return dev / ns if ns else 0.0

def clean_wiki(txt):
    if not txt: return ''
    txt = re.sub(r'<!--.*?-->',                          '',  txt, flags=re.DOTALL)
    txt = re.sub(r'<ref[^>]*>.*?</ref>',                 '',  txt, flags=re.DOTALL)
    txt = re.sub(r'<ref[^>]*/?>',                        '',  txt)
    txt = re.sub(r'\{\{.*?\}\}',                         '',  txt, flags=re.DOTALL)
    txt = re.sub(r'\[\[(File|Image|\u091a\u093f\u0924\u094d\u0930|\u092b\u093e\u0907\u0932):.*?\]\]', '', txt, flags=re.I|re.DOTALL)
    txt = re.sub(r'\[\[[^\]|]*\|([^\]]+)\]\]',           r'\1', txt)
    txt = re.sub(r'\[\[([^\]]+)\]\]',                    r'\1', txt)
    txt = re.sub(r'==+[^=]+==+|<[^>]+>|\[http[^\]]*\]', '',  txt)
    lines = [re.sub(r'\s+', ' ', l).strip() for l in txt.split('\n')]
    return '\n'.join(l for l in lines if len(l) >= 20 and ne_ratio(l) >= 0.45)

def clean_html(html_content):
    try: soup = BeautifulSoup(html_content, 'lxml')
    except Exception: soup = BeautifulSoup(html_content, 'html.parser')
    for t in soup(['script','style','header','footer','nav','aside','noscript','iframe']):
        t.decompose()
    body = soup.find('main') or soup.find('article') or soup.body or soup
    lines = []
    for p in body.find_all(['p','h1','h2','h3','li']):
        l = re.sub(r'\s+', ' ', p.get_text(' ')).strip()
        if len(l) >= 20 and ne_ratio(l) >= 0.50:
            lines.append(l)
    return '\n'.join(lines)

def write_doc(src, uid, text, done, fail, stats):
    global _save_ctr
    wc = len(text.split())
    try:
        with file_lock:
            with open(TXT_FILE, 'a', encoding='utf-8') as f:
                f.write(f'<<SRC:{src}>>\n<<ID:{uid}>>\n{text}\n<<END>>\n\n')
    except Exception as e:
        print(f'  [write_doc] error: {e}'); return 0
    with state_lock:
        done.add(uid)
        stats[uid] = {'w': wc}
        _save_ctr += 1
        if _save_ctr % 25 == 0:
            flush_ckpt(done, fail, stats)
    return wc

print('Setup complete.')
print(f'Output folder : {OUT_DIR}')
print(f'Text file     : {TXT_FILE}')
done0, _ = load_ckpt(); st0 = load_stats()
print(f'Already saved : {len(done0):,} pages | ~{est_tokens(st0):,} tokens')


## Cell 2: Wikimedia Dumps

In [ ]:

def download_with_resume(url, tmp_path, max_retries=6):
    for attempt in range(1, max_retries + 1):
        existing = os.path.getsize(tmp_path) if os.path.exists(tmp_path) else 0
        headers = dict(HEADERS)
        mode = 'wb'
        if existing > 0:
            headers['Range'] = f'bytes={existing}-'; mode = 'ab'
        try:
            with requests.get(url, headers=headers, stream=True, timeout=120) as r:
                if r.status_code == 416: return True
                r.raise_for_status()
                total = int(r.headers.get('content-length', 0)) + existing
                with tqdm(total=total, initial=existing, unit='B', unit_scale=True, desc="   Download", leave=False) as pbar:
                    with open(tmp_path, mode) as f:
                        for chunk in r.iter_content(chunk_size=4*1024*1024):
                            if chunk:
                                f.write(chunk); pbar.update(len(chunk))
            return True
        except Exception as e:
            print(f'   Attempt {attempt} failed: {e}')
            if attempt < max_retries:
                time.sleep(min(30, 2 ** attempt))
            else:
                return False
    return False

def process_dump(name, url, ns_to_base, min_words=20):
    done, fail = load_ckpt()
    stats = load_stats()
    tmp = os.path.join(OUT_DIR, f'_tmp_{name}.xml.bz2')
    print(f'\n── {name} ──')
    if not download_with_resume(url, tmp):
        print(f'   Giving up on {name}.')
        if os.path.exists(tmp): os.remove(tmp)
        return
    pages, words = 0, 0
    try:
        with bz2.BZ2File(tmp, 'rb') as bz:
            for event, elem in ET.iterparse(bz, events=('end',)):
                try:
                    if not elem.tag.endswith('page'): continue
                    ns_el = elem.find('{*}ns'); ns_val = ns_el.text if ns_el is not None else '0'
                    if ns_val not in ns_to_base: elem.clear(); continue
                    title_el = elem.find('{*}title'); title = title_el.text if title_el is not None else ''
                    uid = f"{ns_to_base[ns_val]}{quote(title.replace(' ', '_'))}"
                    if uid in done: elem.clear(); continue
                    rev = elem.find('{*}revision')
                    if rev is not None:
                        te_el = rev.find('{*}text'); wt = te_el.text if te_el is not None else ''
                        if wt and wt.strip().upper().startswith('#REDIRECT'): elem.clear(); continue
                        txt = clean_wiki(wt)
                        if len(txt.split()) > min_words:
                            write_doc(name, uid, txt, done, fail, stats)
                            pages += 1; words += len(txt.split())
                            if pages % 2000 == 0:
                                flush_ckpt(done, fail, stats)
                                print(f'   {pages:,} articles | ~{est_tokens(stats):,} tokens total')
                except Exception:
                    pass
                finally:
                    elem.clear()
    except Exception as e:
        print(f'   Parse error: {e}')
    finally:
        if os.path.exists(tmp): os.remove(tmp)
    flush_ckpt(done, fail, stats)
    print(f'   ✅ {name}: {pages:,} articles | +{words:,} words | ~{est_tokens(stats):,} total tokens')

DUMPS = [
    ('Nepali_Wikipedia', 'https://dumps.wikimedia.org/newiki/latest/newiki-latest-pages-articles.xml.bz2',
     {'0': 'https://ne.wikipedia.org/wiki/'}, 20),
    ('Nepali_Wikisource', 'https://dumps.wikimedia.org/newikisource/latest/newikisource-latest-pages-articles.xml.bz2',
     {'0': 'https://ne.wikisource.org/wiki/', '104': 'https://ne.wikisource.org/wiki/Page:'}, 15),
    ('Nepali_Wikibooks', 'https://dumps.wikimedia.org/newikibooks/latest/newikibooks-latest-pages-articles.xml.bz2',
     {'0': 'https://ne.wikibooks.org/wiki/'}, 15),
    ('Nepali_Wiktionary', 'https://dumps.wikimedia.org/newiktionary/latest/newiktionary-latest-pages-articles.xml.bz2',
     {'0': 'https://ne.wiktionary.org/wiki/'}, 5),
]

for name, url, ns_map, min_w in DUMPS:
    try:
        process_dump(name, url, ns_map, min_words=min_w)
    except Exception as e:
        print(f'\n❌ {name} skipped: {e}')


## Cell 3: News & Government (parallel discovery + fetch)

In [ ]:

# FASTER: all feeds now discovered AND fetched concurrently instead of one feed fully
# finishing before the next starts. This is also the fix for "stuck on the first URL for
# hours" -- a slow site's sitemap discovery no longer blocks every other site behind it.

FEEDS = [
    ('https://ekantipur.com/sitemap.xml',                        'sitemap'),
    ('https://www.khabarhub.com/sitemap.xml',                    'sitemap'),
    ('https://gorkhapatraonline.com/sitemap.xml',                 'sitemap'),
    ('https://www.dcnepal.com/sitemap.xml',                       'sitemap'),
    ('https://www.souryadaily.com/sitemap.xml',                   'sitemap'),
    ('https://nepalsamaya.com/sitemap.xml',                       'sitemap'),
    ('https://www.deshsanchar.com/sitemap.xml',                   'sitemap'),
    ('https://www.nepalnews.com/sitemap.xml',                     'sitemap'),
    ('https://www.onlinekhabar.com/feed',                        'rss'),
    ('https://www.setopati.com/feed',                            'rss'),
    ('https://www.setopati.com/sitemap.xml',                     'sitemap'),
    ('https://myrepublica.nagariknetwork.com/feed',              'rss'),
    ('https://nagariknews.nagariknetwork.com/feed',              'rss'),
    ('https://www.ratopati.com/feed',                            'rss'),
    ('https://www.ratopati.com/sitemap.xml',                     'sitemap'),
    ('https://kathmandupost.com/sitemap.xml',                    'sitemap'),
    ('https://www.annapurnapost.com/sitemap.xml',                'sitemap'),
    ('https://www.bbc.com/nepali/sitemap.xml',                   'sitemap'),
    ('https://ujyaaloonline.com/sitemap.xml',                    'sitemap'),
    ('https://ujyaaloonline.com/feed',                           'rss'),
    ('https://www.pahilpost.com/sitemap.xml',                    'sitemap'),
    ('https://www.pahilpost.com/feed',                           'rss'),
    ('https://baahrakhari.com/sitemap.xml',                      'sitemap'),
    ('https://baahrakhari.com/feed',                             'rss'),
    ('https://imagekhabar.com/sitemap.xml',                      'sitemap'),
    ('https://imagekhabar.com/feed',                             'rss'),
    ('https://nepalkhabar.com/sitemap.xml',                      'sitemap'),
    ('https://nepalkhabar.com/feed',                             'rss'),
    ('https://lokaantar.com/sitemap.xml',                        'sitemap'),
    ('https://lokaantar.com/feed',                               'rss'),
    ('https://nepal.gov.np/sitemap.xml',                         'sitemap'),
    ('https://moha.gov.np/sitemap.xml',                          'sitemap'),
    ('https://mofa.gov.np/sitemap.xml',                          'sitemap'),
    ('https://opmcm.gov.np/sitemap.xml',                         'sitemap'),
    ('https://moest.gov.np/sitemap.xml',                         'sitemap'),
]

NON_TEXT_EXTS = ('.xml', '.jpg', '.jpeg', '.png', '.gif', '.pdf', '.docx', '.xlsx', '.zip', '.mp4')

def get_urls(session, url, ftype, done, depth=0):
    if depth > 2: return []
    try:
        throttle(url)
        r = session.get(url, headers=HEADERS, timeout=12)
        if r.status_code != 200: return []
        try: soup = BeautifulSoup(r.content, 'lxml-xml')
        except Exception: soup = BeautifulSoup(r.content, 'html.parser')
        if ftype == 'rss':
            out = []
            for item in soup.find_all(['item','entry']):
                lk = item.find('link')
                if not lk: continue
                u = (lk.get('href') or lk.text or '').strip()
                if u.startswith('http') and u not in done: out.append(u)
            return out
        subs = soup.find_all('sitemap')
        if subs:
            out = []
            for sm in subs:
                loc = sm.find('loc')
                if loc and loc.text.strip().startswith('http'):
                    out += get_urls(session, loc.text.strip(), 'sitemap', done, depth+1)
            return out
        return [l.text.strip() for l in soup.find_all('loc')
                if l.text.strip().startswith('http')
                and not any(l.text.strip().lower().endswith(x) for x in NON_TEXT_EXTS)
                and l.text.strip() not in done]
    except Exception:
        return []

def fetch_article(session, url, done, fail, stats, tag='news_gov'):
    if url in done or url in fail: return 0
    try:
        throttle(url)
        r = session.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 404:
            with state_lock: fail.add(url)
            return 0
        if r.status_code in (429, 503):
            time.sleep(5); return 0
        if r.status_code != 200: return 0
        txt = clean_html(r.content)
        if len(txt.split()) > 50:
            return write_doc(tag, url, txt, done, fail, stats)
    except Exception:
        pass
    return 0

done, fail = load_ckpt()
stats = load_stats()
session = requests.Session()
session.max_redirects = 5

print(f'=== News & Government Crawl ({len(FEEDS)} sources, discovering all in parallel) ===')

# Phase 1: discover URLs from ALL feeds concurrently (this is what was blocking on "first URL")
all_urls = []
with ThreadPoolExecutor(max_workers=DISCOVERY_WORKERS) as ex:
    futs = {ex.submit(get_urls, session, feed_url, ftype, done): feed_url for feed_url, ftype in FEEDS}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Discovering feeds", unit="feed"):
        feed_url = futs[fut]
        try:
            urls = fut.result()
            all_urls.extend(u for u in urls if u not in done)
        except Exception as e:
            print(f'  ❌ {feed_url}: {e}')

all_urls = list(dict.fromkeys(all_urls))  # dedup, preserve order
print(f'Discovered {len(all_urls):,} total new article URLs across all sources.\n')

# Phase 2: fetch ALL articles from ALL sites in one big pool -- per-domain throttle keeps
# each site polite individually, while overall throughput scales with how many distinct
# domains are in the mix.
saved = 0
with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as ex:
    futs = [ex.submit(fetch_article, session, u, done, fail, stats) for u in all_urls]
    for i, fut in enumerate(tqdm(as_completed(futs), total=len(futs), desc="Fetching articles", unit="page")):
        try:
            if fut.result() > 0: saved += 1
        except Exception:
            pass
        if (i + 1) % 200 == 0:
            flush_ckpt(done, fail, stats)

flush_ckpt(done, fail, stats)
print(f'\nNews & government crawl complete. Saved {saved:,} pages | Total tokens: ~{est_tokens(stats):,}')


## Cell 4: Live Wikimedia API Fallback

In [ ]:

WIKI_APIS = [
    ('API_Nepali_Wikipedia',  'https://ne.wikipedia.org/w/api.php'),
    ('API_Nepali_Wiktionary', 'https://ne.wiktionary.org/w/api.php'),
    ('API_Nepali_Wikibooks',  'https://ne.wikibooks.org/w/api.php'),
    ('API_Nepali_Wikisource', 'https://ne.wikisource.org/w/api.php'),
]

# uid scheme matches the dump parser's exactly (base_url + quoted title) so the checkpoint
# actually dedups against Cell 2's dump-based content instead of re-saving it under a new ID.
# Wikisource queries BOTH namespace 0 and 104 (Page:), same fix as the dump parser.
WIKI_API_NS = {
    'API_Nepali_Wikipedia':  [('0', 'https://ne.wikipedia.org/wiki/')],
    'API_Nepali_Wiktionary': [('0', 'https://ne.wiktionary.org/wiki/')],
    'API_Nepali_Wikibooks':  [('0', 'https://ne.wikibooks.org/wiki/')],
    'API_Nepali_Wikisource': [('0', 'https://ne.wikisource.org/wiki/'),
                               ('104', 'https://ne.wikisource.org/wiki/Page:')],
}

def fetch_wiki_titles(api_url, session, namespace, apcontinue=None, limit=500):
    params = {'action': 'query', 'list': 'allpages', 'apnamespace': namespace,
              'aplimit': limit, 'format': 'json'}
    if apcontinue: params['apcontinue'] = apcontinue
    r = session.get(api_url, params=params, headers=HEADERS, timeout=20)
    data = r.json()
    titles = [p['title'] for p in data.get('query', {}).get('allpages', [])]
    cont = data.get('continue', {}).get('apcontinue')
    return titles, cont

def fetch_wiki_extract(api_url, session, titles):
    params = {'action': 'query', 'prop': 'extracts', 'explaintext': 1,
              'titles': '|'.join(titles), 'format': 'json'}
    r = session.get(api_url, params=params, headers=HEADERS, timeout=30)
    data = r.json()
    out = {}
    for page in data.get('query', {}).get('pages', {}).values():
        out[page.get('title', '')] = page.get('extract', '')
    return out

def crawl_wiki_api(name, api_url, max_pages=200_000):
    done, fail = load_ckpt()
    stats = load_stats()
    session = requests.Session()
    saved, seen = 0, 0
    for ns, base_url in WIKI_API_NS[name]:
        apcontinue = None
        pbar = tqdm(total=max_pages, desc=f"{name} (ns={ns})", unit="page")
        while seen < max_pages:
            try:
                throttle(api_url, delay=0.2)
                titles, apcontinue = fetch_wiki_titles(api_url, session, ns, apcontinue)
                if not titles: break
                new_titles = [t for t in titles if f"{base_url}{quote(t.replace(' ', '_'))}" not in done]
                for i in range(0, len(new_titles), 20):
                    batch = new_titles[i:i+20]
                    throttle(api_url, delay=0.2)
                    extracts = fetch_wiki_extract(api_url, session, batch)
                    for t, ex in extracts.items():
                        seen += 1
                        pbar.update(1)
                        if not ex: continue
                        wc = len(ex.split())
                        if wc < 25: continue
                        uid = f"{base_url}{quote(t.replace(' ', '_'))}"
                        write_doc(name, uid, ex, done, fail, stats)
                        saved += 1
                if saved and saved % 2000 < 20:
                    flush_ckpt(done, fail, stats)
                if not apcontinue: break
            except Exception:
                time.sleep(2)
        pbar.close()
    flush_ckpt(done, fail, stats)
    print(f'   ✅ {name}: {saved:,} pages saved | ~{est_tokens(stats):,} total tokens')

for name, api_url in WIKI_APIS:
    try:
        crawl_wiki_api(name, api_url)
    except Exception as e:
        print(f'\n❌ {name} execution failed: {e}')


## Cell 5: Nepali Literature (BFS, seeds run concurrently)

In [ ]:

LIT_SEEDS_NEPALI = [
    'https://nepalisahityaghar.com/',
    'https://www.onlinesahitya.com/books',
    'https://nepalikitab.org/',
]
# Unverified sites -- check the first "saved N pages" per seed before trusting a big crawl on it.

def crawl_site_bfs(seed, session, done, fail, stats, allowed_domain, max_pages=3000):
    q, visited, saved = deque([seed]), set(), 0
    pbar = tqdm(total=max_pages, desc=f"BFS: {allowed_domain[:25]}", unit="page", leave=False)
    while q and len(visited) < max_pages:
        url = q.popleft()
        if url in visited: continue
        visited.add(url)
        pbar.update(1)
        try:
            throttle(url)
            r = session.get(url, headers=HEADERS, timeout=12)
            if r.status_code != 200: continue
            if url not in done:
                txt = clean_html(r.content)
                if len(txt.split()) > 50:
                    if write_doc('literature', url, txt, done, fail, stats) > 0:
                        saved += 1
            soup = BeautifulSoup(r.content, 'html.parser')
            for a in soup.find_all('a', href=True):
                nu = urljoin(url, a['href'])
                if urlparse(nu).netloc == allowed_domain and nu not in visited and '#' not in nu:
                    q.append(nu)
        except Exception:
            continue
    pbar.close()
    return saved

done, fail = load_ckpt()
stats = load_stats()
session = requests.Session()

print('=== Nepali Literature Sites (running seeds concurrently) ===')

def _run_seed(seed):
    domain = urlparse(seed).netloc
    try:
        s = crawl_site_bfs(seed, session, done, fail, stats, domain)
        flush_ckpt(done, fail, stats)
        return seed, s, None
    except Exception as e:
        return seed, 0, e

with ThreadPoolExecutor(max_workers=len(LIT_SEEDS_NEPALI)) as ex:
    for seed, saved, err in ex.map(_run_seed, LIT_SEEDS_NEPALI):
        if err:
            print(f'  ❌ {seed}: {err}')
        else:
            print(f'  ✅ {seed} → saved {saved:,} pages | ~{est_tokens(stats):,} total tokens')


## Cell 6: Internet Archive Deep Sweep (parallel item fetch)

In [ ]:

IA_QUERIES_NEPALI = [
    'language:Nepali AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Nepali literature" AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Nepal" AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Nepal history" AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Nepali poetry" AND mediatype:texts AND NOT access-restricted-item:true',
    'creator:"Nepali" AND mediatype:texts AND NOT access-restricted-item:true',
]

API = 'https://archive.org/advancedsearch.php'
ROWS_PER_PAGE = 200
MAX_PAGES_PER_QUERY = 15

done, fail = load_ckpt()
stats = load_stats()
session = requests.Session()
saved_total = 0

def _fetch_archive_item(item):
    iid = item.get('identifier', '')
    key = f'archive_{iid}'
    if key in done or key in fail: return 0
    text = ''
    for tu in [f'https://archive.org/download/{iid}/{iid}.txt',
               f'https://archive.org/stream/{iid}/{iid}_djvu.txt']:
        try:
            throttle(tu, delay=0.3)
            rr = session.get(tu, headers=HEADERS, timeout=15)
            if rr.status_code == 200 and len(rr.text) > 500:
                text = rr.text; break
        except Exception:
            pass
    if not text:
        with state_lock: fail.add(key)
        return 0
    lines = [l.strip() for l in text.split('\n')
             if len(l.strip()) >= 20 and ne_ratio(l.strip()) >= 0.40]
    clean = '\n'.join(lines)
    if len(clean.split()) > 100:
        write_doc('archive_downloaded_ocr', key, clean, done, fail, stats)
        return 1
    return 0

print('=== Internet Archive Deep Sweep (parallel item fetch) ===')
for q in tqdm(IA_QUERIES_NEPALI, desc="Queries", unit="query"):
    for page in range(1, MAX_PAGES_PER_QUERY + 1):
        try:
            resp = session.get(API, params={
                'q': q, 'fl[]': 'identifier,title',
                'rows': ROWS_PER_PAGE, 'page': page, 'output': 'json', 'sort[]': 'downloads desc'
            }, timeout=20)
            if resp.status_code != 200: break
            items = resp.json().get('response', {}).get('docs', [])
            if not items: break

            with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as ex:
                futs = [ex.submit(_fetch_archive_item, it) for it in items]
                for fut in tqdm(as_completed(futs), total=len(futs), desc=f"  ↳ Page {page}", leave=False, unit="book"):
                    try:
                        saved_total += fut.result()
                    except Exception:
                        pass

            flush_ckpt(done, fail, stats)
        except Exception as e:
            print(f'\n    Page {page} error: {e} (continuing)')
            break

flush_ckpt(done, fail, stats)
print(f'Archive done. Saved {saved_total} books this run. Total tokens: ~{est_tokens(stats):,}')


## Cell 7: Final Summary

In [ ]:

done, fail = load_ckpt()
stats = load_stats()
total_words = sum(s.get('w', 0) for s in stats.values())
total_tok = int(total_words * 1.3)
target = 500_000_000

print('=' * 58)
print('  NEPALI CORPUS SUMMARY')
print('=' * 58)
print(f'  Unique pages / items saved  : {len(done):,}')
print(f'  Failed / skipped URLs       : {len(fail):,}')
print(f'  Total clean Nepali words    : {total_words:,}')
print(f'  Estimated tokens            : {total_tok:,}')
print(f'  Target                      : {target:,}')
print(f'  Progress toward target      : {total_tok/target*100:.1f}%')
if os.path.exists(TXT_FILE):
    mb = os.path.getsize(TXT_FILE) / 1024 / 1024
    print(f'  Output file size            : {mb:.1f} MB')
print(f'  Output file                 : {TXT_FILE}')
print('=' * 58)
